### Step 1: Import libraries and API Keys

In [22]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import display, Markdown
import gradio as gr
import json

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception("API key not found")

### Step 2: Setup Pushover

In [23]:
pushover_user= os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

print(f"PUSHOVER_USER: {pushover_user}")
print(f"PUSHOVER_TOKEN: {pushover_token}")

PUSHOVER_USER: usirayudtxm7k5nn6jj39qczvaqibj
PUSHOVER_TOKEN: a7macpjd2i18up2bmekhwf53p94ewm


In [24]:
# Test Pushover API

import requests
def send_notification(message: str):
    payload = {
        "user": pushover_user,
        "token": pushover_token,
        "message": message
    }
    requests.post(pushover_url, data=payload)

### Step 3: Describe Pushover as as LLM tool

In [26]:
send_notification_function = {
    "name": "send_notification",
    "description": "Send a push notifcation to the user phone using pushover. Use this to alert the user about important events or time sensitive information.",
    "parameters": {
        "type": "object",
        "properties": {
            "message": {
                "type": "string",
                "description": "The notifiication message to send to the user device"
            }
        },
        "required": ["message"]
    }
}

### Step 4: Add Pushover to the list of tools for the LLM

In [27]:
tools = [{
    "type": "function",
    "function": send_notification_function
}]

### Step 5: Calling the tool from an LLM

In [38]:
def handle_tool_call(tool_calls):
    tool_call = tool_calls[0]
    args = json.loads(tool_call.function.arguments)

    # Actually send the notification using the tool
    send_notification(args["message"])
    #print(f" Sent notification: {args['message']}")

    tool_call_result = {
        "role": "tool",
        "content": f"Notification sent: {args['message']}",
        "tool_call_id": tool_call.id 
    }
    print(f"Tool call result: {tool_call_result}")
    
    # return what to add to our "context" (about tool call results), a dictionary.
    return tool_call_result

In [39]:
client = OpenAI()
messages=[
        {"role": "user", "content": "Please send me a notification telling me what amazing progress i am making in the AI engineering course."}
    ]
response = client.chat.completions.create(
    model= "gpt-4o-mini",
    messages=messages,
    tools=tools
)

# Check if model wants to call a tool
message = response.choices[0].message

if message.tool_calls:
    # Handle tool call
    tool_result = handle_tool_call(message.tool_calls)
    # add message to "context"
    messages.append(message)
    # Add info about the tool call response to the "context", i.e. messages.
    messages.append(tool_result)
    # Invoke the LLM one more time to get its updated response.
    response = client.chat.completions.create(
        model = "gpt-4o-mini",
        messages = messages
    )
    message = response.choices[0].message
    
print(message.content)

Tool call result: {'role': 'tool', 'content': "Notification sent: You're making amazing progress in your AI engineering course! Keep up the great work!", 'tool_call_id': 'call_W84chljt51vvTO7t06E7Trid'}
I've sent you a notification: "You're making amazing progress in your AI engineering course! Keep up the great work!" Keep pushing forward!
